# 🫁 3D Medical Reconstruction — MONAI + PyVista Pipeline

**Cải tiến so với pipeline cũ (skimage MC):**
- ✅ MONAI `Spacing` → resample về isotropic 1×1×1 mm → mesh không bị bẹp trục Z
- ✅ VTK Marching Cubes (C++) → nhanh hơn ~15x so với skimage
- ✅ Taubin smooth bằng VTK C++ → nhanh hơn ~1000x so với vòng for Python
- ✅ Quadric decimation VTK → topology tốt hơn
- ✅ Export thêm file `.glb` (3D universal) để mở bằng Windows 3D Viewer / Blender


In [ ]:
# ============================================================
# CELL 1 — Cài đặt thư viện
# Chạy 1 lần, ~2–3 phút trên Kaggle
# ============================================================
!pip install -q monai pyvista nibabel trimesh plotly
print('✅ Libraries installed')

In [ ]:
# ============================================================
# CELL 2 — Import
# ============================================================
import os
import numpy as np
import nibabel as nib
import pyvista as pv
import trimesh
import plotly.graph_objects as go
import warnings

from monai.transforms import (
    LoadImage,
    EnsureChannelFirst,
    Spacing,
)

warnings.filterwarnings('ignore')

# PyVista chạy headless trên Kaggle (không cần display)
pv.global_theme.jupyter_backend = 'none'

print('✅ Imports OK')

In [ ]:
# ============================================================
# CELL 3 — Cấu hình đường dẫn
# Đổi NII_PATH về đúng file của bạn
# ============================================================
NII_PATH   = "/kaggle/input/datasets/khoayakult/data-khoa/2_case_00066_pred.nii"
OUTPUT_DIR = "/kaggle/working"

case_name  = os.path.basename(NII_PATH).replace('.nii.gz','').replace('.nii','')
HTML_OUT   = os.path.join(OUTPUT_DIR, f"3d_{case_name}_monai_pyvista.html")
GLB_OUT    = os.path.join(OUTPUT_DIR, f"3d_{case_name}_monai_pyvista.glb")

assert os.path.exists(NII_PATH), f"❌ Không tìm thấy file: {NII_PATH}"
print(f"✅ File: {NII_PATH}")
print(f"   Size: {os.path.getsize(NII_PATH)/1e6:.1f} MB")

In [ ]:
# ============================================================
# CELL 4 — Load NIfTI + MONAI Isotropic Resampling
#
# TẠI SAO CẦN BƯỚC NÀY?
#   CT scan thường có spacing ví dụ [0.78, 0.78, 3.0] mm
#   → trục Z thưa hơn 4x so với X,Y
#   → nếu chạy MC trực tiếp, mesh bị "bẹp" theo Z
#   MONAI Spacing resample về 1×1×1 mm trước khi extract surface
# ============================================================
print('Loading NIfTI file...')
nii         = nib.load(NII_PATH)
spacing_orig = np.array(nii.header.get_zooms())
print(f"  Original shape  : {nii.get_fdata(dtype=np.float32).shape}")
print(f"  Original spacing: {spacing_orig} mm")

print('\nRunning MONAI isotropic resampling...')
loader = LoadImage(image_only=True)
img    = loader(NII_PATH)
img    = EnsureChannelFirst()(img)           # [C, X, Y, Z]

# Resample về 1×1×1 mm, mode='nearest' giữ đúng label integer
img_iso = Spacing(pixdim=(1.0, 1.0, 1.0), mode='nearest')(img)
seg_iso = img_iso[0].numpy().astype(np.uint8)   # [X, Y, Z]

print(f"  Isotropic shape : {seg_iso.shape}")
print(f"  Unique labels   : {np.unique(seg_iso).tolist()}")

# Phân phối nhãn
label_map = {0:'Background', 1:'Kidney', 2:'Tumor/Mass', 3:'Cyst', 4:'Label4', 5:'Label5'}
for lab in np.unique(seg_iso):
    cnt = int((seg_iso == lab).sum())
    pct = cnt / seg_iso.size * 100
    print(f"  Label {lab} ({label_map.get(lab, '?')}): {cnt:,} voxels ({pct:.2f}%)")

In [ ]:
# ============================================================
# CELL 5 — Auto-crop để tiết kiệm RAM
# ============================================================
fg = np.where(seg_iso > 0)
if len(fg[0]) == 0:
    raise ValueError('❌ Segmentation rỗng! Kiểm tra lại file.')

MARGIN = 15   # voxels — tăng lên 15 vì spacing đã 1mm, cần margin lớn hơn
x0 = max(0, fg[0].min()-MARGIN);  x1 = min(seg_iso.shape[0], fg[0].max()+MARGIN)
y0 = max(0, fg[1].min()-MARGIN);  y1 = min(seg_iso.shape[1], fg[1].max()+MARGIN)
z0 = max(0, fg[2].min()-MARGIN);  z1 = min(seg_iso.shape[2], fg[2].max()+MARGIN)

cropped = seg_iso[x0:x1, y0:y1, z0:z1]
print(f'Shape sau isotropic: {seg_iso.shape}')
print(f'Shape sau crop     : {cropped.shape}')
print(f'RAM usage ước tính : {cropped.nbytes / 1e6:.1f} MB')

In [ ]:
# ============================================================
# CELL 6 — Hàm tiện ích: PyVista mesh → Plotly Mesh3d
# ============================================================
def pyvista_to_plotly(pv_mesh, color, opacity, name):
    """
    Chuyển PyVista PolyData sang Plotly Mesh3d trace.
    PyVista lưu faces dạng [3, i, j, k, 3, i2, j2, k2, ...]
    → reshape (-1, 4) rồi bỏ cột đầu (số 3).
    """
    verts = np.array(pv_mesh.points)
    
    # Đảm bảo toàn bộ là tam giác
    tri_mesh = pv_mesh.triangulate()
    faces_flat = tri_mesh.faces.reshape(-1, 4)[:, 1:]  # (N, 3)
    
    return go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces_flat[:, 0], j=faces_flat[:, 1], k=faces_flat[:, 2],
        color=color,
        opacity=opacity,
        name=name,
        showscale=False,
        flatshading=False,
        lighting=dict(
            ambient=0.45,
            diffuse=0.85,
            specular=0.35,
            roughness=0.45,
            fresnel=0.25,
        ),
        lightposition=dict(x=100, y=200, z=300),
    )


def mesh_stats(pv_mesh, name):
    """In và trả về stats của mesh."""
    is_mani = pv_mesh.is_manifold
    vol     = round(float(pv_mesh.volume), 2) if is_mani else None
    sa      = round(float(pv_mesh.area), 2)
    stats = {
        'name': name,
        'vertices': pv_mesh.n_points,
        'faces': pv_mesh.n_faces,
        'surface_area_mm2': sa,
        'volume_mm3': vol,
        'is_manifold': is_mani,
    }
    print(f"  Stats → {stats}")
    return stats


print('✅ Utility functions ready')

In [ ]:
# ============================================================
# CELL 7 — Build mesh từng label bằng PyVista + VTK
#
# PIPELINE MỖI LABEL:
#   1. Tạo PyVista ImageData (point data)
#   2. contour([0.5])  → VTK Marching Cubes (C++)
#   3. smooth_taubin() → VTK Taubin smooth (C++)
#   4. decimate()      → VTK Quadric decimation
#   5. Lọc connected components nhỏ
# ============================================================
PARTS = [
    # label, name, color hex, opacity, smooth_iter, pass_band, decimate_ratio, min_faces
    # pass_band: nhỏ hơn = mượt hơn (0.01 rất mượt, 0.1 vừa phải)
    # decimate : 0.7 = xóa 70% tam giác, giữ 30%
    {'label': 1, 'name': 'Than (Kidney)',  'color': '#E8735A', 'opacity': 0.35,
     'smooth_iter': 30, 'pass_band': 0.05, 'decimate': 0.70, 'min_faces': 200},

    {'label': 2, 'name': 'Khoi U (Tumor)', 'color': '#FFD700', 'opacity': 0.95,
     'smooth_iter': 20, 'pass_band': 0.08, 'decimate': 0.50, 'min_faces': 30},

    {'label': 3, 'name': 'Nang (Cyst)',    'color': '#00CED1', 'opacity': 0.90,
     'smooth_iter': 20, 'pass_band': 0.08, 'decimate': 0.50, 'min_faces': 30},

    {'label': 4, 'name': 'Label 4',        'color': '#AA00FF', 'opacity': 0.90,
     'smooth_iter': 15, 'pass_band': 0.10, 'decimate': 0.50, 'min_faces': 20},

    {'label': 5, 'name': 'Label 5',        'color': '#00FF88', 'opacity': 0.90,
     'smooth_iter': 15, 'pass_band': 0.10, 'decimate': 0.50, 'min_faces': 20},
]

fig          = go.Figure()
summary      = []
meshes_all   = []   # Lưu để export GLB
colors_all   = []   # Màu tương ứng cho GLB

for part in PARTS:
    lab  = part['label']
    name = part['name']
    print(f"\n{'='*55}")
    print(f"  {name}  (label={lab})")
    print(f"{'='*55}")

    mask  = (cropped == lab).astype(np.uint8)
    n_vox = int(mask.sum())

    if n_vox == 0:
        print(f"  ⚠️  Không có voxel nào cho {name}. Bỏ qua.")
        continue

    print(f"  Voxels: {n_vox:,}")

    # ----------------------------------------------------------
    # BƯỚC 1 — Tạo PyVista ImageData
    # Spacing đã là 1×1×1 mm sau MONAI → không cần điều chỉnh thêm
    # ----------------------------------------------------------
    grid = pv.ImageData()
    grid.dimensions = mask.shape               # (nx, ny, nz) — point data
    grid.spacing    = (1.0, 1.0, 1.0)         # mm/voxel, isotropic
    grid.origin     = (0.0, 0.0, 0.0)
    # Lưu vào point_data để contour() hoạt động đúng
    grid.point_data['values'] = mask.flatten(order='F').astype(np.float32)

    # ----------------------------------------------------------
    # BƯỚC 2 — VTK Marching Cubes
    # contour() gọi vtkContourFilter (MC bên dưới)
    # level=0.5 → biên giữa 0 và 1
    # ----------------------------------------------------------
    try:
        mesh = grid.contour([0.5], scalars='values')
    except Exception as e:
        print(f"  ❌ MC Error: {e}")
        continue

    if mesh.n_points == 0:
        print("  ⚠️  Mesh rỗng sau MC. Bỏ qua.")
        continue

    print(f"  [MC]      → {mesh.n_points:,} verts | {mesh.n_faces:,} faces")

    # ----------------------------------------------------------
    # BƯỚC 3 — VTK Taubin Smoothing
    # pass_band nhỏ = mượt hơn nhưng mất chi tiết nhỏ
    # normalize_coordinates=True → ổn định hơn cho shape bất thường
    # ----------------------------------------------------------
    mesh = mesh.smooth_taubin(
        n_iter=part['smooth_iter'],
        pass_band=part['pass_band'],
        normalize_coordinates=True,
        feature_smoothing=False,   # Giữ edge features
    )
    print(f"  [Taubin]  → {mesh.n_points:,} verts | {mesh.n_faces:,} faces")

    # ----------------------------------------------------------
    # BƯỚC 4 — VTK Quadric Decimation
    # decimate(0.7) = xóa 70%, giữ 30%
    # ----------------------------------------------------------
    mesh = mesh.decimate(part['decimate'])
    print(f"  [Decimate]→ {mesh.n_points:,} verts | {mesh.n_faces:,} faces")

    # ----------------------------------------------------------
    # BƯỚC 5 — Lọc connected components nhỏ
    # connectivity('all') gán RegionId cho từng component
    # Giữ lại các component có đủ cells
    # ----------------------------------------------------------
    labeled     = mesh.connectivity('all')    # thêm scalar 'RegionId'
    region_ids  = np.unique(labeled['RegionId'])
    large_parts = []

    for rid in region_ids:
        component = labeled.threshold(
            value=(rid - 0.5, rid + 0.5),
            scalars='RegionId',
            preference='cell',
        )
        if component.n_faces >= part['min_faces']:
            large_parts.append(component)

    if not large_parts:
        print(f"  ⚠️  Không có component nào >= {part['min_faces']} faces. Giữ lớn nhất.")
        mesh = labeled.connectivity('largest')
    elif len(large_parts) == 1:
        mesh = large_parts[0]
    else:
        # Merge nhiều components
        mesh = large_parts[0].merge(large_parts[1:])

    # Triangulate để đảm bảo toàn tam giác trước khi render
    mesh = mesh.triangulate()
    print(f"  [Filter]  → {mesh.n_points:,} verts | {mesh.n_faces:,} faces | "
          f"{len(large_parts) if large_parts else 1} component(s)")

    # ----------------------------------------------------------
    # Stats
    # ----------------------------------------------------------
    s = mesh_stats(mesh, name)
    summary.append(s)

    # Thêm vào Plotly figure
    fig.add_trace(pyvista_to_plotly(mesh, part['color'], part['opacity'], name))

    # Lưu để export GLB
    meshes_all.append(mesh)
    colors_all.append(part['color'])

    print(f"  ✅ Đã thêm {name} vào model 3D.")

print(f"\n{'='*55}")
print(f"Hoàn thành tất cả labels.")

In [ ]:
# ============================================================
# CELL 8 — Summary bảng kết quả
# ============================================================
print('\n📊 SUMMARY')
print(f"{'─'*70}")
print(f"{'Tên':<20} {'Verts':>8} {'Faces':>8} {'SA (mm²)':>12} {'Vol (mm³)':>12} {'Watertight':>10}")
print(f"{'─'*70}")
for s in summary:
    vol_str = f"{s['volume_mm3']:,.0f}" if s['volume_mm3'] else 'N/A'
    print(f"{s['name']:<20} {s['vertices']:>8,} {s['faces']:>8,} "
          f"{s['surface_area_mm2']:>12,.1f} {vol_str:>12} {str(s['is_manifold']):>10}")
print(f"{'─'*70}")
print('Ghi chú: Volume chỉ có khi mesh là watertight (is_manifold=True)')

In [ ]:
# ============================================================
# CELL 9 — Layout Plotly + Export HTML
# ============================================================
fig.update_layout(
    title=dict(
        text=f'3D Reconstruction — MONAI+PyVista | {case_name}',
        x=0.5,
        font=dict(size=16),
    ),
    scene=dict(
        xaxis=dict(title='X (mm)', showbackground=True, backgroundcolor='#2a2a3e'),
        yaxis=dict(title='Y (mm)', showbackground=True, backgroundcolor='#252535'),
        zaxis=dict(title='Z (mm)', showbackground=True, backgroundcolor='#2a2a3e'),
        aspectmode='data',   # Giữ đúng tỉ lệ vật lý
        bgcolor='#12121f',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2),
            up=dict(x=0, y=0, z=1),
        ),
    ),
    paper_bgcolor='#12121f',
    font=dict(color='white'),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor='rgba(0,0,0,0.6)',
        font=dict(color='white', size=12),
        bordercolor='rgba(255,255,255,0.2)',
        borderwidth=1,
    ),
    width=1100, height=900,
    margin=dict(r=10, l=10, b=10, t=55),
)

fig.write_html(HTML_OUT, include_plotlyjs='cdn', full_html=True)
size_mb = os.path.getsize(HTML_OUT) / 1e6
print(f'✅ HTML saved : {HTML_OUT}')
print(f'   File size  : {size_mb:.1f} MB')

In [ ]:
# ============================================================
# CELL 10 — Export GLB (bonus)
#
# GLB = binary glTF, mở được bằng:
#   - Windows 3D Viewer
#   - Blender (File > Import > glTF)
#   - Three.js / Babylon.js trên web
#   - Online viewer: https://gltf-viewer.donmccurdy.com
#
# Dùng trimesh để ghép tất cả meshes và export GLB
# ============================================================
if meshes_all:
    import trimesh

    # Hex color → RGB float (0–1)
    def hex_to_rgb(h):
        h = h.lstrip('#')
        return [int(h[i:i+2], 16)/255.0 for i in (0, 2, 4)]

    scene = trimesh.scene.Scene()

    for pv_mesh, color_hex in zip(meshes_all, colors_all):
        verts = np.array(pv_mesh.points)
        faces = pv_mesh.faces.reshape(-1, 4)[:, 1:]   # (N, 3)
        rgb   = hex_to_rgb(color_hex)

        tm_mesh = trimesh.Trimesh(
            vertices=verts,
            faces=faces,
            process=False,
        )
        # Gán màu
        tm_mesh.visual = trimesh.visual.ColorVisuals(
            mesh=tm_mesh,
            vertex_colors=np.array([rgb + [1.0]] * len(verts)),  # RGBA
        )
        scene.add_geometry(tm_mesh)

    glb_bytes = scene.export(file_type='glb')
    with open(GLB_OUT, 'wb') as f:
        f.write(glb_bytes)

    print(f'✅ GLB saved  : {GLB_OUT}')
    print(f'   File size  : {os.path.getsize(GLB_OUT)/1e6:.1f} MB')
    print('   Mở bằng   : Windows 3D Viewer hoặc https://gltf-viewer.donmccurdy.com')
else:
    print('⚠️  Không có mesh để export GLB.')

In [ ]:
# ============================================================
# CELL 11 — Liệt kê output files
# ============================================================
print('\n📁 Output files:')
for f in os.listdir(OUTPUT_DIR):
    if f.startswith('3d_'):
        fp   = os.path.join(OUTPUT_DIR, f)
        size = os.path.getsize(fp) / 1e6
        print(f'   {f:<50}  {size:.1f} MB')

print('\n✅ Xong! Tải file HTML và GLB từ Kaggle Output tab.')